## Exercise

Add a tool to set the price of a ticket!

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>

# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

## Initial Setup

In [1]:
import os
import json
import sqlite3
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown
import gradio as gr

# Load environment variables in a file called .env

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:3]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key not set
Google API Key exists and begins AIzaSyDF
Groq API Key exists and begins gsk_
Grok API Key exists and begins xai-
OpenRouter API Key exists and begins sk-


## Establishing OpenAI clients and the models that can be used.

In [2]:
# Connect Google Gemini
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)

# For Groq
groq_url = "https://api.groq.com/openai/v1"
groq = OpenAI(api_key=groq_api_key, base_url=groq_url) 
GPT_MODEL="openai/gpt-oss-120b"

# For OpenRouter
openrouter_url = "https://openrouter.ai/api/v1"
openrouter = OpenAI(base_url=openrouter_url, api_key=openrouter_api_key)
# The models from router
GEMINI_ROUTER = "google/gemini-2.5-flash-lite"
GPT_ROUTER = "openai/gpt-oss-120b"
GEMMA_ROUTER = "google/gemma-3n-e2b-it:free"

# For Ollam
ollama_url = "http://localhost:11434/v1"
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
LAMMA_MODEL="llama3.2"
LAMMA_1B_MODEL="llama3.2:1b"
LAMMA_GEMMA = "gemma3:4b"
LAMMA_GPT_MODEL = "gpt-oss:20b"

## Defining the functions

In [3]:

def get_ticket_price(destination_city)->str:
    """
    Look up the price for a city in the 'prices' table (SQLite).

    Args:
        destination_city (str): Destination city for which ticket price is to be looked up
    Return:
        (str): A sentence containing the result of the query for the LLM to use.
    """
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {destination_city} is ${result[0]}" if result else "No price data available for this city"
    
def get_ticket_discount(destination_city)->str:
    """
    Look up the maximum discount for a city in the 'prices' table (SQLite).

    Args:
        destination_city (str): Destination city for which ticket maximum discount is to be looked up
    Return:
        (str): A sentence containing the result of the query for the LLM to use.
    """
    print(f"DATABASE TOOL CALLED: Getting price for {destination_city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT max_discount FROM prices WHERE city = ?', (destination_city.lower(),))
        result = cursor.fetchone()
        return f"Ticket maximun discount to {destination_city} is ${result[0]}" if result else "No price data available for this city"


def set_ticket_price(city: str, price: float, max_discount: float = 0.0, user: str = "Not_Authorized")->str:
    """
    Insert or update the price of a city in the 'prices' table (SQLite), if the user is authorized.

    Args:
        city (str): Destination city for which ticket price is to be updated or inserted.
        prince (float): The price value to be set/update. 
        max_discount (float): The maximum discount that can be offered to the customer.
        user: user that request this operation.
    Return:
        (str): A sentence containing the result of the query for the LLM to use.
    """
    if city:
        print(f"DATABASE TOOL CALLED: Setting price for {city}", flush=True)
    else:
        return ""
    
    if user != "ADM_MGM":
        result = "It has been detected that you requested a price change that you are not authorized to make. Instead, you can check if discounts are available."
    elif max_discount > 0.2:
        result = "You cannot apply a discount greater than 20%!"
    else:
        query = """
            INSERT INTO prices (city, price, max_discount)
            VALUES (?, ?, ?)
            ON CONFLICT(city)
            DO UPDATE SET price = excluded.price;
        """
        with sqlite3.connect(DB) as conn:
            cursor = conn.cursor()
            result = cursor.execute(
                query, 
                (city.lower(), price, max_discount)
                )
            conn.commit()

            result = "Done!"
    
    return result


## Mapping the functions used by the LLM Assistent

In [4]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

set_price_function = {  
    "name": "set_ticket_price",
    "description": "Set or update the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {  
                "type": "string",
                "description": "Destination city (e.g. London, Paris)"
            },
            "price": {  
                "type": "number",
                "description": "Ticket price (numeric), e.g. 899"
            },
            "max_discount": {  
                "type": "number",
                "description": "The maximum discount (numeric) that can be offered to the customer, e.g. 0.12"
            },
            "user": {  
                "type": "string",
                "description": "The user login that request a insert or a update on price table"
            }
        },
        "required": ["city", "price", "user"],     
        "additionalProperties": False      
    }
}

discount_function = {
    "name": "get_ticket_discount",
    "description": "Get the maximum discount that can be offered to the customer. Call this whenever you need to know the discount on the ticket price.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The maximum discount that can be offered to the customer is if they travel with the same airline for both the outbound and return flights.",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

# And this is included in a list of tools:
tools = [  
    {"type": "function", "function": price_function},  
    {"type": "function", "function": discount_function},  
    {"type": "function", "function": set_price_function}, 
]

TOOL_MAP = {
    "get_ticket_price":    lambda destination_city, **_: get_ticket_price(destination_city),
    "get_ticket_discount": lambda destination_city, **_: get_ticket_discount(destination_city),
    "set_ticket_price": lambda **_: set_ticket_price(**_),
}

## Create the database and load the initial data.

In [5]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    # cursor.execute("DROP TABLE prices;")
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL, max_discount REAL)')
    conn.commit()
    
ticket_prices = [("london", 799, 0.05) , ("paris", 899, 0.1), ("tokyo", 1420, 0.15), ("sydney", 2999, 0.2), ("berlin", 820, 0.05)]
for city, price, max_discount in ticket_prices:
    set_ticket_price(city, price, max_discount, "ADM_MGM")

DATABASE TOOL CALLED: Setting price for london
DATABASE TOOL CALLED: Setting price for paris
DATABASE TOOL CALLED: Setting price for tokyo
DATABASE TOOL CALLED: Setting price for sydney
DATABASE TOOL CALLED: Setting price for berlin


## Setting the system prompt

In [6]:
system_prompt = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
Request the user loging if he/she request a insert of new city and its price or a update on a price.
Please note that our prices are at least on average 1 percent lower than the competition.
Do not offering any discount, wait for the client ask fo it.
You only can offering a discout to the customer if he/she get a round-trip ticket.
You need to decide what discount you will give
You do not need to inform of the maximum discount, but of a smaller one.
Only offering the maximum discount if you feel that clients are still in doubt.
If the customer closes the deal, inform them of the final price and conditions. 
If it's a round trip, the price of the ticket to the destination city will be double, after applying any discounts you may have granted.
"""

## The Chat and it handeler

In [7]:
def handle_tool_calls(message):
    # Handle ALL tool calls in this response
    tool_messages = []

    if message.tool_calls:
        for tool_call in message.tool_calls:
            fn_name = tool_call.function.name
            arguments = json.loads(tool_call.function.arguments)
            print("Called: ", fn_name, "\n. Args: ", arguments)
            result = TOOL_MAP[fn_name](**arguments)

            tool_messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

    return tool_messages

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=GPT_MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = groq.chat.completions.create(model=GPT_MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

## Start Gradio and have fun

In [8]:
gr.ChatInterface(
    fn=chat, 
    type="messages", 
    title="Flight Booking AI Assistant", 
    description="Tell me your destination, and I'll help you find flights at the best prices.").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


Called:  set_ticket_price 
. Args:  {'city': 'Madrid', 'max_discount': 0.12, 'price': 869.75, 'user': 'MGM_ADM'}
DATABASE TOOL CALLED: Setting price for Madrid
Called:  set_ticket_price 
. Args:  {'city': 'Madrid', 'max_discount': 0.12, 'price': 869.75, 'user': 'ADM_MGM'}
DATABASE TOOL CALLED: Setting price for Madrid
Called:  set_ticket_price 
. Args:  {'city': 'Toledo', 'max_discount': 0.2, 'price': 880, 'user': 'MGM_ADM'}
DATABASE TOOL CALLED: Setting price for Toledo
Called:  set_ticket_price 
. Args:  {'city': 'Toledo', 'max_discount': 0.2, 'price': 880, 'user': 'MGM_ADM'}
DATABASE TOOL CALLED: Setting price for Toledo
Called:  get_ticket_price 
. Args:  {'destination_city': 'Madrid'}
DATABASE TOOL CALLED: Getting price for Madrid
Called:  get_ticket_price 
. Args:  {'destination_city': 'Toledo'}
DATABASE TOOL CALLED: Getting price for Toledo
Called:  get_ticket_price 
. Args:  {'destination_city': 'Barcelona'}
DATABASE TOOL CALLED: Getting price for Barcelona
Called:  get_ticket_